# Error Analysis — LR / SVM vs MARBERT (runs on CPU, no GPU needed)

**Files to upload (3):** `test.csv`, `test_predictions.json` (from the baseline notebook), `marbert_test_predictions.json` (from the MARBERT notebook).

Goal: move from *"MARBERT scores higher"* to *"here is WHERE each model fails and why"*.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

needed = ["test.csv", "test_predictions.json", "marbert_test_predictions.json"]
while not all(os.path.exists(f) for f in needed):
    print("Missing:", [f for f in needed if not os.path.exists(f)], "-> upload them (one at a time is fine)")
    from google.colab import files
    files.upload()

test_df = pd.read_csv("test.csv")
base = json.load(open("test_predictions.json", encoding="utf-8"))
mar  = json.load(open("marbert_test_predictions.json", encoding="utf-8"))
SEEDS = [42, 43, 44]
print("Test rows:", len(test_df))

In [ ]:
# Safety check: all prediction files must refer to the SAME 31 messages in the SAME order
for task in ["intent", "sentiment"]:
    truth = test_df[task].tolist()
    assert base[f"{task}__Logistic Regression"]["y_test"] == truth, f"baseline order mismatch ({task})"
    for s in SEEDS:
        assert mar[f"{task}__MARBERT__seed{s}"]["y_test"] == truth, f"MARBERT order mismatch ({task}, seed {s})"
print("OK: same test messages, same order in all files")

In [ ]:
# Confusion matrices (row-normalised = per-class recall). MARBERT is pooled over the 3 seeds.
def cm_norm(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels).astype(float)
    rows = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, rows, out=np.zeros_like(cm), where=rows > 0)

for task in ["intent", "sentiment"]:
    labels = sorted(test_df[task].unique())
    truth = test_df[task].tolist()
    lr_pred = base[f"{task}__Logistic Regression"]["y_pred"]
    m_true = truth * len(SEEDS)
    m_pred = sum([mar[f"{task}__MARBERT__seed{s}"]["y_pred"] for s in SEEDS], [])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for ax, (name, yt, yp) in zip(axes, [("Logistic Regression", truth, lr_pred),
                                         ("MARBERT (3 seeds pooled)", m_true, m_pred)]):
        cm = cm_norm(yt, yp, labels)
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=40, ha="right"); ax.set_yticklabels(labels)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{task.capitalize()} - {name}")
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{cm[i, j]*100:.0f}%", ha="center", va="center",
                        color="white" if cm[i, j] > 0.5 else "black", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"confusion_{task}.png", dpi=150)
    plt.show()

In [ ]:
# Which messages does MARBERT get wrong? (>= 1 of 3 seeds) and how do LR / SVM do on the SAME messages?
pd.set_option("display.max_colwidth", 80)
all_errors = []
for task in ["intent", "sentiment"]:
    d = pd.DataFrame({"text": test_df["text"], "true": test_df[task]})
    for s in SEEDS:
        d[f"MARBERT_s{s}"] = mar[f"{task}__MARBERT__seed{s}"]["y_pred"]
    d["LR"]  = base[f"{task}__Logistic Regression"]["y_pred"]
    d["SVM"] = base[f"{task}__SVM (LinearSVC)"]["y_pred"]
    d["MARBERT_seeds_wrong"] = sum((d[f"MARBERT_s{s}"] != d["true"]).astype(int) for s in SEEDS)
    d["LR_wrong"]  = (d["LR"]  != d["true"]).astype(int)
    d["SVM_wrong"] = (d["SVM"] != d["true"]).astype(int)

    print("=" * 80)
    print(task.upper())
    print("=" * 80)
    print("MARBERT wrong per seed:", [int((d[f"MARBERT_s{s}"] != d["true"]).sum()) for s in SEEDS],
          "| LR wrong:", int(d.LR_wrong.sum()), "| SVM wrong:", int(d.SVM_wrong.sum()))
    print("LR wrong but MARBERT right in all 3 seeds:", int(((d.MARBERT_seeds_wrong == 0) & (d.LR_wrong == 1)).sum()))
    print("LR right but MARBERT wrong in >=1 seed   :", int(((d.MARBERT_seeds_wrong > 0) & (d.LR_wrong == 0)).sum()))
    hard = d[d.MARBERT_seeds_wrong > 0].copy()
    hard.insert(0, "task", task)
    display(hard.drop(columns=["task"]))
    all_errors.append(hard)

pd.concat(all_errors).to_csv("marbert_error_analysis.csv", index=False, encoding="utf-8-sig")
print("Saved: marbert_error_analysis.csv, confusion_intent.png, confusion_sentiment.png")

### What to write in the report
- Report **how many distinct messages** MARBERT fails on (very few), and whether LR/SVM also fail on them.
- Read the failing messages yourself: are they **sarcasm / expectation reversal** ("I expected problems, but..."), **very short slang**, or **questionable labels**?
- Remember: all 3 seeds use the same 31 messages, so this is an *anecdotal* error analysis, not a statistical one.